# TitanAgent Phase 2 — Job 1: CoT 方式比較（`Qwen2.5-0.5B-Instruct` ＋ Titans MAC）

`Qwen/Qwen2.5-0.5B-Instruct` を完全凍結し、Titans Neural Memory（MAC アダプタ）上で以下の 3 条件を同シード・同条件で比較します。

1. **`no_memory_turn_cot`**: Titans 記憶なし（ベースライン）＋ 従来ターン型 CoT
2. **`frozen_turn_cot`**: Titans MAC 記憶あり ＋ 従来ターン型 CoT（`<think>...</think>`）
3. **`frozen_stream_cot`**: Titans MAC 記憶あり ＋ 細粒度インターリーブ・保持型 CoT（フレーズごとに `<t>...</t>` を挟んで保持）

評価は通常距離（ディストラクタ 1〜3）と未見長距離（ディストラクタ 7）の 2 軸で行います。


In [ ]:
# 1. 環境確認＆必要ライブラリ導入
import os, sys, subprocess, torch

IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or os.path.exists('/kaggle/working')
print(f'Environment: Colab={IN_COLAB}, Kaggle={IN_KAGGLE}')
print(f'PyTorch: {torch.__version__}, CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: GPU が無効です！設定で GPU (T4 / P100) を ON にしてください。')

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=4.45.0', 'accelerate'])


In [ ]:
# 2. ハイパーパラメータ設定（Job 1: CoT 方式比較）
from pathlib import Path

MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'
MEM_DIM = 256
NUM_PERSIST = 4
NUM_LONGTERM = 8
TRAIN_STEPS = 300
GRAD_ACCUM = 4
LR = 1e-3
LORA_ALPHA = 16.0
LORA_LR = 2e-4
EVAL_EPISODES = 20
LONG_EVAL_EPISODES = 10
MAX_NEW_TOKENS = 200
LOG_EVERY = 30
SEED = 42

CONDITIONS = [
    ('no_memory_turn_cot', 'turn_cot', False, 0),
    ('frozen_turn_cot',    'turn_cot', True,  0),
    ('frozen_stream_cot',  'stream_cot', True, 0),
]

OUTPUT_DIR = Path('/kaggle/working/checkpoints/phase2_job1_cot_comparison') if IN_KAGGLE else Path('checkpoints/phase2_job1_cot_comparison')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('OUTPUT_DIR =', OUTPUT_DIR)


In [ ]:
# 3. Titans (MAC) Neural Memory Adapter + Pure-PyTorch LoRA + 日本語データ生成器
from dataclasses import asdict, dataclass
import gc, json, math, random, re, time
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


@dataclass
class TitansMemoryConfig:
    hidden_size: int = 896
    mem_dim: int = 256
    num_persist_mem_tokens: int = 4
    num_longterm_mem_tokens: int = 8
    max_grad_norm: float = 1.0
    base_lr_scale: float = 0.15


@dataclass
class TitansMemoryState:
    w1: torch.Tensor
    w2: torch.Tensor
    s1: torch.Tensor
    s2: torch.Tensor
    num_updates: int = 0
    last_surprise: float = 0.0
    last_forget_rate: float = 0.0


class TitansMACAdapter(nn.Module):
    def __init__(self, cfg: TitansMemoryConfig):
        super().__init__()
        self.cfg = cfg
        h, d = cfg.hidden_size, cfg.mem_dim
        self.in_norm = nn.LayerNorm(h)
        self.k_proj = nn.Linear(h, d, bias=False)
        self.v_proj = nn.Linear(h, d, bias=False)
        self.q_proj = nn.Linear(h, d, bias=False)
        self.gate_proj = nn.Linear(h, 3)
        nn.init.zeros_(self.gate_proj.weight)
        with torch.no_grad():
            self.gate_proj.bias.copy_(torch.tensor([-3.0, 0.0, -1.0]))
        self.w1_init = nn.Parameter(torch.empty(d, d))
        self.w2_init = nn.Parameter(torch.empty(d, d))
        nn.init.normal_(self.w1_init, 0.0, 1.0 / math.sqrt(d))
        nn.init.normal_(self.w2_init, 0.0, 1.0 / math.sqrt(d))
        self.persist_mem = nn.Parameter(torch.randn(1, cfg.num_persist_mem_tokens, h) * 0.02)
        self.read_queries = nn.Parameter(torch.randn(1, cfg.num_longterm_mem_tokens, d) * 0.02)
        self.out_norm = nn.LayerNorm(d)
        self.out_proj = nn.Linear(d, h, bias=False)
        self.out_scale = nn.Parameter(torch.tensor(0.1))

    def init_state(self, batch_size: int, dev: torch.device) -> TitansMemoryState:
        w1 = self.w1_init.float().unsqueeze(0).expand(batch_size, -1, -1).clone()
        w2 = self.w2_init.float().unsqueeze(0).expand(batch_size, -1, -1).clone()
        return TitansMemoryState(w1=w1, w2=w2, s1=torch.zeros_like(w1), s2=torch.zeros_like(w2))

    def _mem_fwd(self, x, w1, w2):
        z = torch.bmm(x, w1)
        sig = torch.sigmoid(z)
        h = z * sig
        return torch.bmm(h, w2), z, sig, h

    def update_memory(self, segment_embeds, state: TitansMemoryState):
        x = self.in_norm(segment_embeds.float())
        k = F.normalize(self.k_proj(x), dim=-1)
        v = self.v_proj(x)
        gates = self.gate_proj(x.mean(dim=1))
        alpha = torch.sigmoid(gates[:, 0]).view(-1, 1, 1)
        eta = torch.sigmoid(gates[:, 1]).view(-1, 1, 1)
        theta = F.softplus(gates[:, 2]).view(-1, 1, 1) * self.cfg.base_lr_scale
        pred, z, sig, h = self._mem_fwd(k, state.w1, state.w2)
        diff = pred - v
        surprise = float((0.5 * diff.pow(2).mean()).detach().item())
        err = diff / max(1, k.shape[1])
        dw2 = torch.bmm(h.transpose(1, 2), err)
        dh = torch.bmm(err, state.w2.transpose(1, 2))
        dz = dh * (sig * (1.0 + z * (1.0 - sig)))
        dw1 = torch.bmm(k.transpose(1, 2), dz)
        max_n = self.cfg.max_grad_norm
        dw1 = dw1 * (max_n / dw1.flatten(1).norm(dim=1).view(-1, 1, 1).clamp_min(1e-6)).clamp(max=1.0)
        dw2 = dw2 * (max_n / dw2.flatten(1).norm(dim=1).view(-1, 1, 1).clamp_min(1e-6)).clamp(max=1.0)
        ns1 = eta * state.s1 - theta * dw1
        ns2 = eta * state.s2 - theta * dw2
        nw1 = (1.0 - alpha) * state.w1 + ns1
        nw2 = (1.0 - alpha) * state.w2 + ns2
        fg = float(alpha.mean().detach().item())
        return TitansMemoryState(nw1, nw2, ns1, ns2, state.num_updates + 1, surprise, fg), {
            'surprise': surprise, 'forget_rate': fg
        }

    def retrieve_prefix(self, prompt_embeds, state: TitansMemoryState, out_dtype):
        bsz = prompt_embeds.shape[0]
        pq = self.q_proj(self.in_norm(prompt_embeds.float()).mean(dim=1, keepdim=True))
        queries = F.normalize(self.read_queries.float().expand(bsz, -1, -1) + pq, dim=-1)
        ret, _, _, _ = self._mem_fwd(queries, state.w1, state.w2)
        ret = self.out_proj(self.out_norm(ret)) * self.out_scale
        persist = self.persist_mem.float().expand(bsz, -1, -1)
        return torch.cat([persist, ret], dim=1).to(dtype=out_dtype)


class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, r: int = 8, alpha: float = 16.0):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad = False
        self.r = r
        self.alpha = alpha
        self.scaling = alpha / float(r)
        dev = base.weight.device
        self.lora_A = nn.Parameter(torch.empty((r, base.in_features), dtype=torch.float32, device=dev))
        self.lora_B = nn.Parameter(torch.zeros((base.out_features, r), dtype=torch.float32, device=dev))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))

    def forward(self, x):
        base_out = self.base(x)
        lora_out = F.linear(F.linear(x.float(), self.lora_A), self.lora_B) * self.scaling
        return base_out + lora_out.to(dtype=base_out.dtype)


def apply_lora_to_model(model, r=8, alpha=16.0, targets=('q_proj', 'k_proj', 'v_proj', 'o_proj')):
    if r <= 0:
        return 0
    count = 0
    for _, module in list(model.named_modules()):
        for child_name, child in list(module.named_children()):
            if isinstance(child, nn.Linear) and child_name in targets:
                setattr(module, child_name, LoRALinear(child, r=r, alpha=alpha))
                count += 1
    return count


class TitansMACLLM(nn.Module):
    def __init__(self, base_model, mem_cfg: TitansMemoryConfig, use_memory=True, lora_r=0, lora_alpha=16.0):
        super().__init__()
        self.base_model = base_model
        for p in self.base_model.parameters():
            p.requires_grad = False
        self.lora_r = lora_r
        self.lora_alpha = lora_alpha
        self.lora_layers_wrapped = apply_lora_to_model(self.base_model, r=lora_r, alpha=lora_alpha) if lora_r > 0 else 0
        self.use_memory = use_memory
        self.mem_cfg = mem_cfg
        self.adapter = TitansMACAdapter(mem_cfg)

    def lora_parameters(self):
        return [p for n, p in self.base_model.named_parameters() if p.requires_grad and ('lora_A' in n or 'lora_B' in n)]

    def lora_state_dict(self):
        return {k: v.detach().cpu() for k, v in self.base_model.state_dict().items() if 'lora_A' in k or 'lora_B' in k}

    @property
    def prefix_len(self):
        return self.mem_cfg.num_persist_mem_tokens + self.mem_cfg.num_longterm_mem_tokens

    def embed_ids(self, ids):
        return self.base_model.get_input_embeddings()(ids)

    def memorize_segment(self, seg_ids, state):
        if not self.use_memory:
            return state, {'surprise': 0.0, 'forget_rate': 0.0}
        with torch.no_grad():
            emb = self.embed_ids(seg_ids)
        return self.adapter.update_memory(emb, state)

    def _get_prefix(self, emb, state):
        if not self.use_memory:
            bsz = emb.shape[0]
            persist = self.adapter.persist_mem.float().expand(bsz, -1, -1)
            zeros = torch.zeros((bsz, self.mem_cfg.num_longterm_mem_tokens, self.mem_cfg.hidden_size), dtype=torch.float32, device=emb.device)
            return torch.cat([persist, zeros], dim=1).to(dtype=emb.dtype)
        return self.adapter.retrieve_prefix(emb, state, emb.dtype)

    def forward_with_memory(self, mem_segs, target_ids, labels):
        state = self.adapter.init_state(target_ids.shape[0], target_ids.device)
        stats = {'surprise': 0.0, 'forget_rate': 0.0}
        if self.use_memory:
            for seg in mem_segs:
                state, stats = self.memorize_segment(seg, state)
        with torch.no_grad():
            t_emb = self.embed_ids(target_ids)
        prefix = self._get_prefix(t_emb, state)
        out = self.base_model(inputs_embeds=torch.cat([prefix, t_emb], dim=1))
        shift_logits = out.logits[:, self.prefix_len : -1, :].float().contiguous()
        shift_labels = labels[:, 1:].contiguous()
        loss = F.cross_entropy(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1), ignore_index=-100)
        return loss, state, stats

    @torch.no_grad()
    def generate_with_memory(self, prompt_ids, state, eos_id, max_new_tokens=200):
        p_emb = self.embed_ids(torch.tensor([prompt_ids], dtype=torch.long, device=device))
        cur = torch.cat([self._get_prefix(p_emb, state), p_emb], dim=1)
        gen = []
        for _ in range(max_new_tokens):
            logits = self.base_model(inputs_embeds=cur).logits[:, -1, :].float()
            nid = int(torch.argmax(logits, dim=-1).item())
            gen.append(nid)
            if nid == eos_id:
                break
            cur = torch.cat([cur, self.embed_ids(torch.tensor([[nid]], dtype=torch.long, device=device))], dim=1)
        return gen


NAMES = ['ケンタ', 'ユウキ', 'ミサキ', 'ハルカ', 'タクミ', 'アオイ', 'リョウ', 'サクラ']
THEMES = ['英語学習', 'アプリ開発', '副業ブログ', '資格勉強', '運動習慣', 'AI研究', 'デザイン学習', '自動化ツール作り']
GOALS = ['毎日続けること', '3か月以内に形にすること', '基礎を固めること', '小さく公開すること', '収入につなげること']
CONSTRAINTS = ['平日は30分だけ', '週末にまとめて作業', '予算は無料枠のみ', '夜しか時間がない', 'スマホ中心で進めたい']
STYLES = ['簡潔に要点だけ', '丁寧にステップ順で', '具体例を交えて', '背中を押す感じで']
DISTRACTORS = [
    'ユーザー: 今日は少し天気がいいですね。ちょっと一息ついてます。\nアシスタント: いいですね、無理のないペースでリフレッシュしてください。',
    'ユーザー: さっきコーヒーを淹れました。作業環境を整えています。\nアシスタント: 環境が整うと集中しやすいですよね。準備ができたら進みましょう。',
    'ユーザー: そういえば昨日読んだ記事が面白かったです。\nアシスタント: 良い刺激になりますね。気になった視点があればメモしておくと役立ちます。',
    'ユーザー: ちょっと別の用事を片付けてきます。\nアシスタント: 了解です！戻ってきたらまた続きを話しましょう。',
    'ユーザー: メモの整理をしていました。少しずつ見通しが立ってきました。\nアシスタント: 整理が進んで何よりです。今の流れを大事にしましょう。',
]

THINK_RE = re.compile(r'<think>.*?</think>\s*', flags=re.DOTALL)
STREAM_RE = re.compile(r'<t>.*?</t>', flags=re.DOTALL)


def strip_cot(text: str) -> str:
    return STREAM_RE.sub('', THINK_RE.sub('', text)).strip()


def gen_episode(rng, stage=2, cot_mode='turn_cot', force_overwrite=None, num_dist_override=None):
    latest = {
        '名前': rng.choice(NAMES),
        'テーマ': rng.choice(THEMES),
        '目標': rng.choice(GOALS),
        '制約': rng.choice(CONSTRAINTS),
        '好み': rng.choice(STYLES),
    }
    stale = {}
    if cot_mode == 'stream_cot':
        seg1 = (
            f"ユーザー: はじめまして、{latest['名前']}です。今取り組んでいるのは{latest['テーマ']}で、"
            f"目標は「{latest['目標']}」、制約は「{latest['制約']}」です。アドバイスは{latest['好み']}お願いします。\n"
            f"アシスタント: <t>記憶:名前={latest['名前']}</t>{latest['名前']}さん、よろしくお願いします！"
            f"<t>記憶:テーマ={latest['テーマ']}|制約={latest['制約']}</t>{latest['テーマ']}を「{latest['制約']}」の条件で進められるように一緒に考えていきましょう。"
        )
    else:
        seg1 = (
            f"ユーザー: はじめまして、{latest['名前']}です。今取り組んでいるのは{latest['テーマ']}で、"
            f"目標は「{latest['目標']}」、制約は「{latest['制約']}」です。アドバイスは{latest['好み']}お願いします。\n"
            f"アシスタント: <think>記憶: 名前={latest['名前']} / テーマ={latest['テーマ']} / 目標={latest['目標']} / 制約={latest['制約']} / 好み={latest['好み']}</think>\n"
            f"{latest['名前']}さん、よろしくお願いします！{latest['テーマ']}を「{latest['制約']}」の条件で進められるように一緒に考えていきましょう。"
        )
    segs = [seg1]
    do_ov = (False if stage == 1 else rng.random() < 0.55) if force_overwrite is None else force_overwrite
    n_dist = (rng.randint(0, 1) if stage == 1 else rng.randint(1, 3)) if num_dist_override is None else num_dist_override
    for _ in range(n_dist):
        segs.append(rng.choice(DISTRACTORS))
    if do_ov:
        slot = rng.choice(['制約', '目標', 'テーマ'])
        old_v = latest[slot]
        pool = CONSTRAINTS if slot == '制約' else (GOALS if slot == '目標' else THEMES)
        new_v = rng.choice([x for x in pool if x != old_v])
        stale[slot] = old_v
        latest[slot] = new_v
        if cot_mode == 'stream_cot':
            segs.append(
                f"ユーザー: ちょっと状況が変わりました。{slot}を「{new_v}」に変更したいです。\n"
                f"アシスタント: <t>更新:{slot}={new_v}|旧:{old_v}破棄</t>了解しました！{slot}を「{new_v}」に更新して覚えておきますね。"
            )
        else:
            segs.append(
                f"ユーザー: ちょっと状況が変わりました。{slot}を「{new_v}」に変更したいです。\n"
                f"アシスタント: <think>更新記憶: {slot}={new_v}（旧:{old_v}は破棄）</think>\n"
                f"了解しました！{slot}を「{new_v}」に更新して覚えておきますね。"
            )
        if stage == 2:
            segs.append(rng.choice(DISTRACTORS))

    prompt = 'ユーザー: 前に伝えた私の名前と、今のテーマ・目標・制約を踏まえて、次の一歩を提案してくれますか？\nアシスタント: '
    if cot_mode == 'stream_cot':
        target = (
            f"<t>名前:{latest['名前']}</t>{latest['名前']}さん、"
            f"<t>テーマ:{latest['テーマ']}</t>現在のテーマは{latest['テーマ']}、"
            f"<t>目標:{latest['目標']}</t>目標は「{latest['目標']}」、"
            f"<t>制約:{latest['制約']}</t>制約は「{latest['制約']}」ですね。"
            f"<t>好み:{latest['好み']}</t>{latest['好み']}提案します。"
            f"<t>次の一歩:{latest['制約']}+{latest['テーマ']}->{latest['目標']}</t>"
            f"まずは「{latest['制約']}」の中で{latest['テーマ']}を小さく1回試し、「{latest['目標']}」へつなげましょう。"
        )
    else:
        target = (
            f"<think>記憶リハーサル: 名前={latest['名前']} | テーマ={latest['テーマ']} | 目標={latest['目標']} | 制約={latest['制約']} | 好み={latest['好み']}</think>\n"
            f"{latest['名前']}さん、現在の前提（テーマ: {latest['テーマ']} / 目標: {latest['目標']} / 制約: {latest['制約']}）に合わせて{latest['好み']}提案します。"
            f"まずは「{latest['制約']}」の中で{latest['テーマ']}を小さく1回試し、「{latest['目標']}」へつなげましょう。"
        )
    return segs, prompt, target, latest, stale, do_ov


def score_reply(reply, latest, stale, do_ov, cot_mode):
    vis = strip_cot(reply)
    rec_vis = sum(1 for k in ['名前', 'テーマ', '目標', '制約'] if latest[k] in vis) / 4.0
    rec_full = sum(1 for k in ['名前', 'テーマ', '目標', '制約'] if latest[k] in reply) / 4.0
    ov_ok, used_stale = True, False
    if do_ov:
        for k, old_v in stale.items():
            if latest[k] not in vis:
                ov_ok = False
            if old_v in vis and latest[k] not in vis:
                used_stale = True
    fmt_ok = (reply.count('<t>') >= 4 and reply.count('</t>') >= 4) if cot_mode == 'stream_cot' else ('<think>' in reply and '</think>' in reply)
    return {'recall_rate': rec_vis, 'full_recall_rate': rec_full, 'overwrite_ok': ov_ok, 'used_stale': used_stale, 'format_ok': fmt_ok}


def run_condition(name, cot_mode, use_memory, lora_r, tokenizer, output_dir):
    print(f'\n==================== [{name}] cot={cot_mode} | mem={use_memory} | lora_r={lora_r} ====================')
    torch.manual_seed(SEED)
    rng = random.Random(SEED)
    dtype = torch.float16 if device.type == 'cuda' else torch.float32
    base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=dtype, trust_remote_code=True).to(device)
    base_model.eval()

    mem_cfg = TitansMemoryConfig(
        hidden_size=int(base_model.config.hidden_size),
        mem_dim=MEM_DIM,
        num_persist_mem_tokens=NUM_PERSIST,
        num_longterm_mem_tokens=NUM_LONGTERM,
    )
    wrapper = TitansMACLLM(base_model, mem_cfg, use_memory=use_memory, lora_r=lora_r, lora_alpha=LORA_ALPHA).to(device)
    param_groups = [{'params': list(wrapper.adapter.parameters()), 'lr': LR, 'base_lr': LR}]
    lora_params = wrapper.lora_parameters()
    if lora_params:
        param_groups.append({'params': lora_params, 'lr': LORA_LR, 'base_lr': LORA_LR})
    optimizer = torch.optim.AdamW(param_groups, weight_decay=0.01)
    all_trainable = [p for g in param_groups for p in g['params']]
    print(f'Trainable params: {sum(p.numel() for p in all_trainable)/1e6:.2f}M (lora_layers={wrapper.lora_layers_wrapped})')

    stage1_steps = max(1, int(TRAIN_STEPS * 0.30))
    warmup_steps = max(1, int(TRAIN_STEPS * 0.05))
    wrapper.adapter.train()
    t0 = time.time()
    last_loss = 0.0

    for step in range(1, TRAIN_STEPS + 1):
        stage = 1 if step <= stage1_steps else 2
        if step <= warmup_steps:
            scale = step / warmup_steps
        else:
            prog = (step - warmup_steps) / max(1, TRAIN_STEPS - warmup_steps)
            scale = 0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * prog))
        for pg in optimizer.param_groups:
            pg['lr'] = pg['base_lr'] * scale

        optimizer.zero_grad(set_to_none=True)
        aloss, asurp = 0.0, 0.0
        for _ in range(GRAD_ACCUM):
            segs, prompt, target, _, _, _ = gen_episode(rng, stage=stage, cot_mode=cot_mode)
            mem_t = [torch.tensor([tokenizer.encode(s, add_special_tokens=False)], dtype=torch.long, device=device) for s in segs]
            p_ids = tokenizer.encode(prompt, add_special_tokens=False)
            r_ids = tokenizer.encode(target, add_special_tokens=False) + [tokenizer.eos_token_id]
            full_t = torch.tensor([p_ids + r_ids], dtype=torch.long, device=device)
            lbl_t = torch.tensor([[-100] * len(p_ids) + r_ids], dtype=torch.long, device=device)
            loss, _, st = wrapper.forward_with_memory(mem_t, full_t, lbl_t)
            (loss / GRAD_ACCUM).backward()
            aloss += float(loss.item()) / GRAD_ACCUM
            asurp += st['surprise'] / GRAD_ACCUM

        torch.nn.utils.clip_grad_norm_(all_trainable, 1.0)
        optimizer.step()
        last_loss = aloss
        if step == 1 or step % LOG_EVERY == 0 or step == TRAIN_STEPS:
            print(f'  [{name} | S{stage}] step {step:4d}/{TRAIN_STEPS} | loss {aloss:.4f} | surprise {asurp:.4f} | {time.time()-t0:.1f}s')

    train_secs = round(time.time() - t0, 1)
    ckpt_path = output_dir / f'{name}.pt'
    torch.save({
        'mem_cfg': asdict(mem_cfg),
        'adapter_state_dict': wrapper.adapter.state_dict(),
        'lora_r': lora_r,
        'lora_state_dict': wrapper.lora_state_dict(),
    }, ckpt_path)

    def _eval(n_eps, seed_off, dist_override=None):
        wrapper.adapter.eval()
        erng = random.Random(SEED + seed_off)
        et0 = time.time()
        rec_s, full_s, ov_ok, ov_tot, stale_c, fmt_c, tok_s = 0.0, 0.0, 0, 0, 0, 0, 0
        sample = {}
        for i in range(n_eps):
            segs, prompt, _, latest, stale, do_ov = gen_episode(erng, stage=2, cot_mode=cot_mode, force_overwrite=(i % 2 == 1), num_dist_override=dist_override)
            st = wrapper.adapter.init_state(1, device)
            for s in segs:
                st, _ = wrapper.memorize_segment(torch.tensor([tokenizer.encode(s, add_special_tokens=False)], dtype=torch.long, device=device), st)
            gids = wrapper.generate_with_memory(tokenizer.encode(prompt, add_special_tokens=False), st, tokenizer.eos_token_id, max_new_tokens=MAX_NEW_TOKENS)
            tok_s += len(gids)
            reply = tokenizer.decode(gids, skip_special_tokens=True)
            sc = score_reply(reply, latest, stale, do_ov, cot_mode)
            rec_s += sc['recall_rate']
            full_s += sc['full_recall_rate']
            fmt_c += int(sc['format_ok'])
            if do_ov:
                ov_tot += 1
                ov_ok += int(sc['overwrite_ok'])
                stale_c += int(sc['used_stale'])
            if i == 1:
                sample = {'latest': latest, 'stale': stale, 'reply': reply[:360], 'score': sc}
                print(f'  [Sample {name} | d={dist_override or "1-3"}] Latest={latest} | Stale={stale}')
                print(f'    Reply: {reply[:260]}')
        return {
            'episodes': n_eps,
            'recall_rate': round(rec_s / max(1, n_eps), 4),
            'full_recall_rate': round(full_s / max(1, n_eps), 4),
            'overwrite_rate': round(ov_ok / max(1, ov_tot), 4),
            'stale_rate': round(stale_c / max(1, ov_tot), 4),
            'format_ok_rate': round(fmt_c / max(1, n_eps), 4),
            'avg_gen_tokens': round(tok_s / max(1, n_eps), 1),
            'eval_seconds': round(time.time() - et0, 1),
            'sample': sample,
        }

    ev_std = _eval(EVAL_EPISODES, 999, None)
    ev_long = _eval(LONG_EVAL_EPISODES, 1999, 7)

    del wrapper, base_model, optimizer
    gc.collect()
    if device.type == 'cuda':
        torch.cuda.empty_cache()

    return {
        'config': {'cot_mode': cot_mode, 'use_memory': use_memory, 'lora_r': lora_r},
        'train': {'steps': TRAIN_STEPS, 'final_loss': round(last_loss, 4), 'train_seconds': train_secs},
        'eval_standard': ev_std,
        'eval_long_d7': ev_long,
    }


In [ ]:
# 4. 全条件を順番に学習＆評価
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

all_results = {
    'job': 'phase2_job1_cot_comparison',
    'model': MODEL_NAME,
    'train_steps': TRAIN_STEPS,
    'grad_accum': GRAD_ACCUM,
    'conditions': {},
}

for cname, cot_mode, use_mem, lora_r in CONDITIONS:
    all_results['conditions'][cname] = run_condition(cname, cot_mode, use_mem, lora_r, tokenizer, OUTPUT_DIR)
    (OUTPUT_DIR / 'metrics.json').write_text(json.dumps(all_results, ensure_ascii=False, indent=2), encoding='utf-8')


In [ ]:
# 5. 最終比較サマリー表示
print('\n=== Phase 2 Experiment Comparison Summary ===')
header = f"{'Condition':24s} | {'Loss':>6s} | {'StdRec':>6s} | {'StdOv':>6s} | {'StdStl':>6s} | {'StdFmt':>6s} | {'L7Rec':>6s} | {'L7Ov':>6s} | {'Toks':>5s}"
print(header)
print('-' * len(header))
for name, res in all_results['conditions'].items():
    es = res['eval_standard']
    el = res['eval_long_d7']
    print(
        f"{name:24s} | {res['train']['final_loss']:6.3f} | "
        f"{es['recall_rate']:6.3f} | {es['overwrite_rate']:6.3f} | {es['stale_rate']:6.3f} | {es['format_ok_rate']:6.3f} | "
        f"{el['recall_rate']:6.3f} | {el['overwrite_rate']:6.3f} | {es['avg_gen_tokens']:5.1f}"
    )
